In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\micah\Desktop\Analyst\Portfolio Projects\transactions_cleaned.csv")

df

,transaction_id,date,customer_id,type,category,amount,plan_tier,region
0,1000023,7/26/2025,615574,expense,Hosting & Infrastructure,230.23,Enterprise,APAC
1,1000197,7/12/2025,496350,revenue,Upsell,213.97,Growth,APAC
2,1000442,10/31/2025,508962,expense,Sales Commissions,228.06,Starter,APAC
3,1000660,2/7/2025,567378,expense,Marketing,462.30,Growth,APAC
4,1000786,12/22/2024,178440,revenue,Upsell,33.66,Starter,Europe
...,...,...,...,...,...,...,...,...
71600,9999564,9/15/2024,506437,expense,Sales Commissions,213.72,Enterprise,Europe
71601,9999610,11/5/2025,227422,expense,Customer Support,297.01,Growth,Europe
71602,9999633,1/14/2025,376141,expense,Salaries & Benefits,883.53,Starter,North America
71603,9999730,4/15/2025,964498,revenue,Subscription,1696.73,Enterprise,North America


In [ ]:
#-----------------------------------------------------------Quick Validation---------------------------------------------------------------------------

In [3]:
#checking NULLs in region, must be 6

df['region'].isnull().sum()

np.int64(6)

In [4]:
#checking NULLs in plan_tier, must be 14

df['plan_tier'].isnull().sum()

np.int64(14)

In [5]:
#checking count of rows, must be 71,605 after deleting duplicates

df['transaction_id'].count()

np.int64(71605)

In [6]:
#confirming there is no more duplicates, must be 0

df.duplicated(subset=['date','customer_id','type','category','amount','plan_tier','region']).sum()

np.int64(0)

In [7]:
#converting to date

df['date'] = pd.to_datetime(df['date']) 

In [8]:
#-----------------------------------------------------------EDA Starts---------------------------------------------------------------------------

In [8]:
#What's the revenue and margin trend over time (overall)?

yearly = df.pivot_table(index=df['date'].dt.year, columns='type', values='amount', aggfunc='sum')  
yearly['margin'] = yearly['revenue'] - yearly['expense']
yearly

type,expense,revenue,margin
date,,,
2023,752137.73,935660.28,183522.55
2024,2533575.47,3860562.10,1326986.63
2025,7160741.62,10924611.81,3763870.19


In [9]:
#Sub Q2

# Which segments (region/plan tier) have the best vs. worst margin, 
# and which segment's revenue and margin growth decelerated the most between 2023–2024 and 2024–2025?

#Which region accelerated/decelerated in revenue/margin between 2023-24, 2024-25? 


region_yearly = df.pivot_table(
    index=['region', df['date'].dt.year],
    columns='type', values='amount', aggfunc='sum'
)
region_yearly['margin'] = region_yearly['revenue'] - region_yearly['expense']
region_yearly = region_yearly.reset_index().rename(columns={'date': 'year'})

region_yearly['revenue_growth'] = region_yearly.groupby('region')['revenue'].pct_change() * 100
region_yearly['margin_growth'] = region_yearly.groupby('region')['margin'].pct_change() * 100

print(region_yearly)

type         region  year     expense     revenue      margin  revenue_growth  \
0              APAC  2023    73223.19   203829.24   130606.05             NaN   
1              APAC  2024   285996.06   961710.50   675714.44      371.821658   
2              APAC  2025   865465.36  2525248.21  1659782.85      162.578833   
3            Europe  2023   220087.99   253407.59    33319.60             NaN   
4            Europe  2024   734314.53  1021004.11   286689.58      302.909838   
5            Europe  2025  2095117.01  2910141.34   815024.33      185.027387   
6     Latin America  2023   257924.50   238286.51   -19637.99             NaN   
7     Latin America  2024   831125.51  1052529.17   221403.66      341.707409   
8     Latin America  2025  2238905.92  2954639.06   715733.14      180.718021   
9     North America  2023   200702.13   240136.94    39434.81             NaN   
10    North America  2024   682139.37   825293.82   143154.45      243.676329   
11    North America  2025  1

In [10]:
#sub Q2 p2
#Which plan_tier accelerated/decelerated revenue/margin between 2023-24, 2024-25?

    
# creating pivot table for revenue+expense of each plan_tier per region
plan_tier_yearly = df.pivot_table(
    index=['plan_tier',df['date'].dt.year], columns=['type'],
    values='amount',aggfunc='sum')

    # creating margin column
plan_tier_yearly['margin'] = plan_tier_yearly['revenue'] - plan_tier_yearly['expense']

plan_tier_yearly = plan_tier_yearly.reset_index().rename(columns={'date':'year'})

    #taking the percentages
plan_tier_yearly['revenue_growth'] = plan_tier_yearly.groupby('plan_tier')['revenue'].pct_change()*100
plan_tier_yearly['margin_growth'] = plan_tier_yearly.groupby('plan_tier')['margin'].pct_change()*100

print(plan_tier_yearly) 

type   plan_tier  year     expense     revenue      margin  revenue_growth  \
0     Enterprise  2023    57626.16   558734.29   501108.13             NaN   
1     Enterprise  2024   234194.85  2457477.40  2223282.55      339.829351   
2     Enterprise  2025   813190.13  7067986.01  6254795.88      187.611435   
3         Growth  2023   218152.56   283395.88    65243.32             NaN   
4         Growth  2024   815282.93  1094210.33   278927.40      286.106647   
5         Growth  2025  2545913.84  3108406.17   562492.33      184.077575   
6        Starter  2023   476359.01    93506.54  -382852.47             NaN   
7        Starter  2024  1483606.47   308833.88 -1174772.59      230.280513   
8        Starter  2025  3801189.80   746514.42 -3054675.38      141.720377   

type  margin_growth  
0               NaN  
1        343.673215  
2        181.331578  
3               NaN  
4        327.518710  
5        101.662630  
6               NaN  
7        206.847332  
8        160.022698  

In [11]:
# Filter to just APAC and Europe, then look at Starter's margin trend within those regions
apac_europe = df[df['region'].isin(['APAC', 'Europe'])]

segment_check = apac_europe.pivot_table(
    index=['plan_tier', apac_europe['date'].dt.year],
    columns='type',
    values='amount',
    aggfunc='sum'
)
segment_check['margin'] = segment_check['revenue'] - segment_check['expense']
segment_check = segment_check.reset_index().rename(columns={'date': 'year'})

print(segment_check)

type   plan_tier  year     expense     revenue      margin
0     Enterprise  2023    15521.78   266775.50   251253.72
1     Enterprise  2024    99926.95  1271338.88  1171411.93
2     Enterprise  2025   335410.48  3424361.93  3088951.45
3         Growth  2023   110543.72   146933.98    36390.26
4         Growth  2024   370307.39   567845.15   197537.76
5         Growth  2025  1107153.65  1648104.74   540951.09
6        Starter  2023   167245.68    43527.35  -123718.33
7        Starter  2024   550076.25   143530.58  -406545.67
8        Starter  2025  1517570.39   362558.49 -1155011.90


In [12]:
# Starter tier margin by region and year, all four regions side by side
starter_by_region = df[df['plan_tier'] == 'Starter'].pivot_table(
    index=['region', df['date'].dt.year],
    columns='type',
    values='amount',
    aggfunc='sum'
)
starter_by_region['margin'] = starter_by_region['revenue'] - starter_by_region['expense']
starter_by_region = starter_by_region.reset_index().rename(columns={'date': 'year'})

print(starter_by_region)

type         region  year     expense    revenue      margin
0              APAC  2023    43080.61   20486.89   -22593.72
1              APAC  2024   171202.34   73479.57   -97722.77
2              APAC  2025   431542.19  173180.26  -258361.93
3            Europe  2023   124165.07   23040.46  -101124.61
4            Europe  2024   378873.91   70051.01  -308822.90
5            Europe  2025  1086028.20  189378.23  -896649.97
6     Latin America  2023   171136.94   25239.76  -145897.18
7     Latin America  2024   521618.38   88207.38  -433411.00
8     Latin America  2025  1229293.73  182769.13 -1046524.60
9     North America  2023   137976.39   24739.43  -113236.96
10    North America  2024   411911.84   77071.42  -334840.42
11    North America  2025  1054325.68  201069.47  -853256.21


In [ ]:
#Is the margin/growth rate drag in that segment driven by rising costs, 
# falling/slowing revenue, or both?

In [13]:
#is growth rate deceleration in APAC/Europe driven by rising costs, 
# falling/slowing in revenue, or both?


# Filter to APAC and Europe only
apac_europe = df[df['region'].isin(['APAC', 'Europe'])]

# Revenue & expense by region and year
apac_europe_yearly = apac_europe.pivot_table(
    index=['region', apac_europe['date'].dt.year],
    columns='type', 
    values='amount',
    aggfunc='sum'
)
apac_europe_yearly['margin'] = apac_europe_yearly['revenue'] - apac_europe_yearly['expense']
apac_europe_yearly = apac_europe_yearly.reset_index().rename(columns={'date': 'year'})

# Growth rates for both revenue and expense, per region
apac_europe_yearly['revenue_growth'] = apac_europe_yearly.groupby('region')['revenue'].pct_change() * 100
apac_europe_yearly['expense_growth'] = apac_europe_yearly.groupby('region')['expense'].pct_change() * 100
apac_europe_yearly['margin_growth'] = apac_europe_yearly.groupby('region')['margin'].pct_change() * 100

print(apac_europe_yearly) 

type  region  year     expense     revenue      margin  revenue_growth  \
0       APAC  2023    73223.19   203829.24   130606.05             NaN   
1       APAC  2024   285996.06   961710.50   675714.44      371.821658   
2       APAC  2025   865465.36  2525248.21  1659782.85      162.578833   
3     Europe  2023   220087.99   253407.59    33319.60             NaN   
4     Europe  2024   734314.53  1021004.11   286689.58      302.909838   
5     Europe  2025  2095117.01  2910141.34   815024.33      185.027387   

type  expense_growth  margin_growth  
0                NaN            NaN  
1         290.581263     417.368407  
2         202.614435     145.633769  
3                NaN            NaN  
4         233.645889     760.423234  
5         185.316023     184.288090  


In [14]:
#Is margin drag in the starter plan driven by rising costs, falling/slowing in revenue, or both?

starter_yearly = df[df['plan_tier']=='Starter'].pivot_table(
    index=df['date'].dt.year,
    columns='type',
    values='amount',
    aggfunc='sum'
)

starter_yearly['margin'] = starter_yearly['revenue'] - starter_yearly['expense']

starter_yearly['revenue growth'] = starter_yearly['revenue'].pct_change()*100
starter_yearly['expense growth'] = starter_yearly['expense'].pct_change()*100
starter_yearly['margin growth'] = starter_yearly['margin'].pct_change()*100 

starter_yearly

type,expense,revenue,margin,revenue growth,expense growth,margin growth
date,,,,,,
2023,476359.01,93506.54,-382852.47,NaN,NaN,NaN
2024,1483606.47,308833.88,-1174772.59,230.280513,211.447131,206.847332
2025,3801189.80,746514.42,-3054675.38,141.720377,156.212808,160.022698


In [ ]:
#Is one outlier segment skewing the overall company-wide numbers, 
#or is the deceleration spread evenly across segments?


In [15]:
# Exclude APAC, recompute overall trend
no_apac = df[df['region'] != 'APAC']
yearly_no_apac = no_apac.pivot_table(index=no_apac['date'].dt.year, columns='type', values='amount', aggfunc='sum')
yearly_no_apac['margin'] = yearly_no_apac['revenue'] - yearly_no_apac['expense']
yearly_no_apac['margin_growth'] = yearly_no_apac['margin'].pct_change() * 100
yearly_no_apac

type,expense,revenue,margin,margin_growth
date,,,,
2023,678914.54,731831.04,52916.50,NaN
2024,2247579.41,2898851.60,651272.19,1130.754472
2025,6295276.26,8399363.60,2104087.34,223.073420


In [16]:
# Exclude Starter, recompute overall trend
no_starter = df[df['plan_tier'] != 'Starter']
yearly_no_starter = no_starter.pivot_table(index=no_starter['date'].dt.year, columns='type', values='amount', aggfunc='sum')
yearly_no_starter['margin'] = yearly_no_starter['revenue'] - yearly_no_starter['expense']
yearly_no_starter['margin_growth'] = yearly_no_starter['margin'].pct_change() * 100
yearly_no_starter

type,expense,revenue,margin,margin_growth
date,,,,
2023,275778.72,842153.74,566375.02,NaN
2024,1049969.00,3551728.22,2501759.22,341.714259
2025,3359551.82,10178097.39,6818545.57,172.550033


In [ ]:
#------------------------------------ALL Four Sub-Questions Done--------------------------------

In [ ]:
#--------------------------------Recommendation Exploration Begin-------------------------------

In [ ]:
#APAC - Expense cateogry expense numbers

apac_expense = df[(df['region'] == 'APAC') & (df['type'] == 'expense')]

apac_expense_by_category = apac_expense.pivot_table(
    index=['category', apac_expense['date'].dt.year],
    values='amount',
    aggfunc='sum'
)
apac_expense_by_category = apac_expense_by_category.reset_index().rename(columns={'date': 'year'})

apac_expense_by_category['growth'] = apac_expense_by_category.groupby('category')['amount'].pct_change() * 100

apac_expense_by_category

,category,year,amount,growth
0,Customer Support,2023,3615.15,NaN
1,Customer Support,2024,17401.70,381.354854
2,Customer Support,2025,55044.76,216.318291
3,G&A,2023,6419.21,NaN
4,G&A,2024,19562.25,204.745444
5,G&A,2025,57256.32,192.687804
6,Hosting & Infrastructure,2023,5589.52,NaN
7,Hosting & Infrastructure,2024,28958.67,418.088673
8,Hosting & Infrastructure,2025,70914.71,144.882483
9,Marketing,2023,11113.04,NaN


In [18]:
#Starter_plan cateogry expense numbers


starter_expense = df[(df['plan_tier'] == 'Starter') & (df['type'] == 'expense')]

starter_expense_by_category = starter_expense.pivot_table(
    index=['category', starter_expense['date'].dt.year],
    values='amount',
    aggfunc='sum'
)
starter_expense_by_category = starter_expense_by_category.reset_index().rename(columns={'date': 'year'})

starter_expense_by_category['growth'] = starter_expense_by_category.groupby('category')['amount'].pct_change() * 100

starter_expense_by_category

,category,year,amount,growth
0,Customer Support,2023,23552.51,NaN
1,Customer Support,2024,101197.30,329.666732
2,Customer Support,2025,285587.87,182.208982
3,G&A,2023,27627.24,NaN
4,G&A,2024,87513.83,216.766459
5,G&A,2025,240143.28,174.406091
6,Hosting & Infrastructure,2023,44677.71,NaN
7,Hosting & Infrastructure,2024,159578.45,257.176879
8,Hosting & Infrastructure,2025,414000.53,159.433858
9,Marketing,2023,68583.20,NaN
